## Import Library and Function

In [2]:
# Function and Library imports
import math
import itertools
from collections import defaultdict
import csv
import time

#Check pairs for validity
def is_valid_pair(pair):
    # Check for invalid vertical pairs (manhatton distance d = 1)
    for state_i, state_j in zip(pair[0], pair[1]):
        if state_i == 1 and state_j == 1:
            return False
    # Check for invalid adjacent pairs (manhatton distance d = 2)
    for i in range(len(pair[0]) - 1):
        pair_upper = (pair[0][i], pair[0][i + 1])
        pair_lower = (pair[1][i], pair[1][i + 1])
        if pair_upper == (1, 0) and pair_lower == (0, 1):
            return False
        elif pair_upper == (0, 1) and pair_lower == (1, 0):
            return False
    return True

def is_valid_triplet(triplet):
    for top_row, middle_row, bottom_row in zip(triplet[0], triplet[1], triplet[2]):
        if top_row == 1 and middle_row == 0 and bottom_row == 1:
            return False
    return True

def maxplus_and_path(matrix_A,matrix_B):
    N = len(matrix_A)
    collect_max = [[-math.inf] * N for _ in range(N)]
    collect_path = [[[] for _ in range(N)] for _ in range(N)]

    for i in range(N):
        for j in range(N):
            best_max = -math.inf
            best_path = []
            for k in range(N):
                if matrix_A[i][k] == -math.inf or matrix_B[k][j] == -math.inf:
                    continue
                path_score = matrix_A[i][k] + matrix_B[k][j]
                if path_score > best_max:
                    best_max = path_score
                    best_path = [k]
                elif path_score == best_max and best_max != -math.inf:
                    best_path.append(k)
            collect_max[i][j] = best_max
            collect_path[i][j] = best_path
    return collect_max, collect_path

all_optimal_paths = []

def backtrack(step_idx, initial_row, temp_row, current_path, path_history_data):
    if step_idx < 0:
        optimal_path = [initial_row, temp_row] + current_path[::-1]
        all_optimal_paths.append(optimal_path)
        return
    previous_rows = path_history_data[step_idx][initial_row][temp_row]
    for row in previous_rows:
        backtrack(step_idx - 1, initial_row, row, current_path + [temp_row], path_history_data)


### Timestamp

In [3]:
timestamp = time.strftime("%Y%m%d-%H%M%S")

## Magnetic Chessgame Setting in Mathematics

### Define size of board

In [4]:
#delta = 2
board_size = 11

### Generate fullshift of board

In [5]:
#Generate states
all_states = list(itertools.product([0, 1], repeat=board_size))

## Define Forbidden Words and Extract Subshift

### Define forbidden words of single states

In [5]:
#Define forbidden words and valid states
forbidden_words = ['11','101']
valid_states = []

#Check all states and filter valid according to forbidden words
for state in all_states:
    state_str = ''.join(map(str, state))
    if not any(word in state_str for word in forbidden_words):
        valid_states.append(state)

print(len(valid_states))

88


In [6]:
with open(f'output/{timestamp}_valid_states.txt', 'w') as f:
    for state in valid_states:
        f.write(str(state) + '\n')
        f.write("-" * 15 + '\n')

print(f"Exported to output/{timestamp}_valid_states.txt")

Exported to output/20260904-134026_valid_states.txt


### Define forbidden words of pair states

In [7]:
pair_states = list(itertools.product(valid_states, repeat=2))
print(len(pair_states))

7744


In [8]:
valid_pairs = [pair for pair in pair_states if is_valid_pair(pair)]

print(len(valid_pairs))

1201


In [9]:
with open(f'output/{timestamp}_valid_pairs.txt', 'w') as f:
    for pair in valid_pairs:
        f.write(str(pair[0]) + '\n')
        f.write(str(pair[1]) + '\n')
        f.write("-" * 15 + '\n')

print(f"Exported to output/{timestamp}_valid_pairs.txt")

Exported to output/20260904-134026_valid_pairs.txt


### Define forbidden words of triplet states

In [10]:
from collections import defaultdict

states_checker = defaultdict(list)
for upper, lower in valid_pairs:
    states_checker[upper].append(lower)

valid_triplets = []
for top_row, middle_row in valid_pairs:
    candidate_bottom_rows = states_checker[middle_row]
    for bottom_row in candidate_bottom_rows:
        if is_valid_triplet((top_row, middle_row, bottom_row)):
            valid_triplets.append((top_row, middle_row, bottom_row))

print(len(valid_triplets))

17894


In [11]:
with open(f'output/{timestamp}_valid_triplets.txt', 'w') as f:
    for triplets in valid_triplets:
        f.write(str(triplets[0]) + '\n')
        f.write(str(triplets[1]) + '\n')
        f.write(str(triplets[2]) + '\n')
        f.write("-" * 15 + '\n')

print(f"Exported to output/{timestamp}_valid_triplets.txt")

Exported to output/20260904-134026_valid_triplets.txt


### Generate transfer matrix of triplet states

In [12]:
adjacency_index = {pair: idx for idx, pair in enumerate(valid_pairs)}

n_valid_pairs = len(valid_pairs)
transfer_matrix = [[-math.inf] * n_valid_pairs for _ in range(n_valid_pairs)]
for top, mid, bot in valid_triplets:
    top_idx = adjacency_index[(top, mid)]
    bot_idx = adjacency_index[(mid, bot)]
    weight = bot.count(1)
    transfer_matrix[top_idx][bot_idx] = weight

print(f"Transfer matrix size: {n_valid_pairs} x {n_valid_pairs}")

Transfer matrix size: 1201 x 1201


In [13]:
try:
    import pandas as pd
    df = pd.DataFrame(transfer_matrix)
    df.to_csv(f'output/{timestamp}_transfer_matrix.csv', index=False)
    print(f"Exported output/{timestamp}_transfer_matrix.csv")
except ImportError:
    for row in transfer_matrix:
        print(" ".join(str(x) for x in row))

Exported output/20260904-134026_transfer_matrix.csv


## Rows Extension

### Transfer Matrix Multiplication

In [14]:
matrix = transfer_matrix
matrix_history = []
path_history = []

for step in range(board_size - 3):
    matrix, path = maxplus_and_path(matrix, transfer_matrix)
    path_history.append(path)
    matrix_history.append(matrix)
    print(f"After step {step + 1}, calculated row: {step + 4} of {board_size}, max score: {max(max(row) for row in matrix)}")

After step 1, calculated row: 4 of 11, max score: 6
After step 2, calculated row: 5 of 11, max score: 8
After step 3, calculated row: 6 of 11, max score: 10
After step 4, calculated row: 7 of 11, max score: 12
After step 5, calculated row: 8 of 11, max score: 14
After step 6, calculated row: 9 of 11, max score: 16
After step 7, calculated row: 10 of 11, max score: 18
After step 8, calculated row: 11 of 11, max score: 20


In [15]:
with open(f'output/{timestamp}_matrix_history.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['step', 'row_index', 'value_row'])
    for step, mat in enumerate(matrix_history, start=1):
        for row_idx, row in enumerate(mat):
            writer.writerow([step, row_idx] + row)
print(f"Exported output/{timestamp}_matrix_history.csv")

with open(f'output/{timestamp}_path_history.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['step', 'row_index', 'path_indices'])
    for step, path in enumerate(path_history, start=1):
        for row_idx, row in enumerate(path):
            writer.writerow([step, row_idx, ";".join(str(k) for k in row)])
print(f"Exported output/{timestamp}_path_history.csv")

Exported output/20260904-134026_matrix_history.csv
Exported output/20260904-134026_path_history.csv


## Maximum Magnets Calculation

### Maximum Score, Optimal Paths Endpoints and Final Matrix Calculation

In [16]:
max_score = -math.inf
best_path_endpoint = []
final_matrix = [[-math.inf] * n_valid_pairs for _ in range(n_valid_pairs)]

for i in range(n_valid_pairs):
    initial_pair = valid_pairs[i]
    initial_score = initial_pair[0].count(1) + initial_pair[1].count(1)
    for j in range(n_valid_pairs):
        if matrix[i][j] != -math.inf:
            total_score = initial_score + matrix[i][j]
            final_matrix[i][j] = total_score
            if total_score > max_score:
                max_score = total_score
                best_path_endpoint = [(i,j)]
            elif total_score == max_score:
                best_path_endpoint.append((i,j))

print(f"Maximum number of magnets is {max_score}")

Maximum number of magnets is 25


In [17]:
try:
    import pandas as pd
    df = pd.DataFrame(final_matrix)
    df.to_csv(f'output/{timestamp}_final_matrix.csv', index=False)
    print(f"Exported output/{timestamp}_final_matrix.csv")
except ImportError:
    for row in final_matrix:
        print(" ".join(str(x) for x in row))

Exported output/20260904-134026_final_matrix.csv


## Optimal Paths Calculation

### Optimal Paths Backtracking

In [18]:
all_optimal_paths.clear()
n_path_history = len(path_history)
for i, j in best_path_endpoint:
    backtrack(n_path_history - 1, i, j, [], path_history)
print(f"Number of optimal paths is {len(all_optimal_paths)}")

Number of optimal paths is 90


In [19]:
with open(f'output/{timestamp}_all_optimal_paths.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    for idx, path in enumerate(all_optimal_paths):
        grid = [valid_pairs[path[0]][0]]
        for p in path:
            grid.append(valid_pairs[p][1])
        writer.writerow([f"Pattern {idx+1}"])
        for row in grid:
            writer.writerow(row)
        writer.writerow([])
print(f"Exported output/{timestamp}_all_optimal_paths.csv")

Exported output/20260904-134026_all_optimal_paths.csv
